# Base Agents Training & Evaluation
## PPO vs Baseline Strategies

**Date:** November 9, 2025  
**Objective:** Train PPO agent, implement baselines, compare performance

---

## Contents
1. Environment Setup & Configuration
2. PPO Agent Training
3. Baseline Strategies Implementation
4. Performance Evaluation
5. Comparative Analysis
6. Portfolio Weights Visualization
7. Risk-Return Analysis

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# RL imports
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.monitor import Monitor
import torch

# Configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

# Project imports
from harlf.config import TICKERS, get_fold_files
from harlf.envs.portfolio_env import PortfolioEnv
from harlf.agents.dirichlet_policy import SoftmaxActorCriticPolicy

print("✅ Imports complete")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

## 1. Environment Setup

Create training, validation, and test environments for Fold 0.

In [ ]:
# Configuration
FOLD_ID = 0
REWARD_TYPE = 'ema_sharpe'
TOTAL_TIMESTEPS = 200000
SEED = 42

# Set seeds
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"📊 Configuration")
print("="*60)
print(f"Fold: {FOLD_ID}")
print(f"Reward: {REWARD_TYPE}")
print(f"Training timesteps: {TOTAL_TIMESTEPS:,}")
print(f"Seed: {SEED}")

In [ ]:
# Create environments
def make_env(fold_id, split, reward_type='ema_sharpe'):
    env = PortfolioEnv(fold_id=fold_id, split=split, reward_type=reward_type)
    env = Monitor(env)
    return env

# Create train, val, test environments
train_env = DummyVecEnv([lambda: make_env(FOLD_ID, 'train', REWARD_TYPE)])
val_env = make_env(FOLD_ID, 'val', REWARD_TYPE)
test_env = make_env(FOLD_ID, 'test', REWARD_TYPE)

print("\n✅ Environments created")
print(f"Train env: {train_env.observation_space.shape}")
print(f"Val env: {val_env.observation_space.shape}")
print(f"Test env: {test_env.observation_space.shape}")

## 2. PPO Agent Training

### Option A: Train New Agent

In [ ]:
# Train new PPO agent (if not already trained)
TRAIN_NEW = True  # Set to True to train from scratch

if TRAIN_NEW:
    print("Training new PPO agent...")
    
    # PPO hyperparameters
    policy_kwargs = {
        'net_arch': [256, 256],
        'activation_fn': torch.nn.Tanh,
        'ortho_init': True,
    }
    
    model = PPO(
        policy=SoftmaxActorCriticPolicy,
        env=train_env,
        learning_rate=3e-4,
        n_steps=2048,
        batch_size=128,
        n_epochs=10,
        gamma=0.985,
        gae_lambda=0.95,
        clip_range=0.2,
        ent_coef=0.01,
        vf_coef=0.5,
        max_grad_norm=0.5,
        policy_kwargs=policy_kwargs,
        verbose=1,
        seed=SEED,
    )
    
    # Train
    model.learn(total_timesteps=TOTAL_TIMESTEPS, progress_bar=True)
    
    # Save
    model.save('../models/fold_0/ppo_fold_0_notebook.zip')
    print("\n✅ Training complete, model saved")
else:
    print("⏩ Skipping training (TRAIN_NEW=False)")

### Option B: Load Existing Model

In [ ]:
# Load pre-trained model
model_path = '../models/fold_0/ppo_fold_0_final.zip'

if Path(model_path).exists():
    print(f"📦 Loading model from {model_path}...")
    model = PPO.load(model_path)
    print("✅ Model loaded successfully")
else:
    print(f"❌ Model not found at {model_path}")
    print("Please train a model first or check the path")

## 3. Baseline Strategies

Implement simple baseline strategies for comparison.

In [ ]:
class EqualWeightStrategy:
    """Equal weight (1/N) strategy."""
    
    def __init__(self, n_assets=7):
        self.n_assets = n_assets
        self.weights = np.ones(n_assets) / n_assets
    
    def predict(self, observation, deterministic=True):
        """Predict action (deterministic parameter ignored for compatibility)."""
        return self.weights, None


class RandomStrategy:
    """Random portfolio weights."""
    
    def __init__(self, n_assets=7, seed=42):
        self.n_assets = n_assets
        self.rng = np.random.RandomState(seed)
    
    def predict(self, observation, deterministic=True):
        """Predict action (deterministic parameter ignored for compatibility)."""
        weights = self.rng.dirichlet(np.ones(self.n_assets))
        return weights, None


class MomentumStrategy:
    """Simple momentum: allocate to assets with positive 21-day returns."""
    
    def __init__(self, n_assets=7):
        self.n_assets = n_assets
    
    def predict(self, observation, deterministic=True):
        """Predict action (deterministic parameter ignored for compatibility)."""
        # Extract return_21d for each asset (feature index 1)
        # Observation is flattened: [asset1_feats, asset2_feats, ...]
        n_features = len(observation) // self.n_assets
        returns = [observation[i * n_features + 1] for i in range(self.n_assets)]
        
        # Allocate to positive momentum assets
        positive_returns = np.array([max(r, 0) for r in returns])
        
        if positive_returns.sum() > 0:
            weights = positive_returns / positive_returns.sum()
        else:
            weights = np.ones(self.n_assets) / self.n_assets
        
        return weights, None


print("✅ Baseline strategies defined")
print("   - Equal Weight (1/N)")
print("   - Random")
print("   - Momentum (21-day)")

## 4. Performance Evaluation

Evaluate all agents on validation and test sets.

In [ ]:
def evaluate_agent(agent, env, agent_name, n_episodes=1, deterministic=True):
    """
    Evaluate an agent on an environment.
    
    Returns:
        Dictionary with performance metrics and episode data
    """
    all_metrics = []
    all_returns = []
    all_weights = []
    all_values = []
    
    for episode in range(n_episodes):
        obs, info = env.reset()
        done = False
        truncated = False
        
        episode_returns = []
        episode_weights = []
        episode_values = [info['portfolio_value']]
        
        while not (done or truncated):
            action, _states = agent.predict(obs, deterministic=deterministic)
            obs, reward, done, truncated, info = env.step(action)
            
            episode_returns.append(info['portfolio_return'])
            episode_weights.append(info['weights'])
            episode_values.append(info['portfolio_value'])
        
        # Get episode metrics from unwrapped env
        unwrapped_env = env.unwrapped if hasattr(env, 'unwrapped') else env
        if hasattr(unwrapped_env, 'get_episode_metrics'):
            metrics = unwrapped_env.get_episode_metrics()
            all_metrics.append(metrics)
        
        all_returns.append(episode_returns)
        all_weights.append(episode_weights)
        all_values.append(episode_values)
    
    # Average metrics across episodes
    if all_metrics:
        avg_metrics = {k: np.mean([m[k] for m in all_metrics]) for k in all_metrics[0].keys()}
    else:
        avg_metrics = {}
    
    return {
        'agent_name': agent_name,
        'metrics': avg_metrics,
        'returns': all_returns[0] if n_episodes == 1 else all_returns,
        'weights': all_weights[0] if n_episodes == 1 else all_weights,
        'values': all_values[0] if n_episodes == 1 else all_values,
    }

print("✅ Evaluation function defined")

In [ ]:
# Evaluate all agents on validation set
print("🔍 Evaluating agents on VALIDATION set...\n")

agents = {
    'PPO': model,
    'Equal Weight': EqualWeightStrategy(),
    'Random': RandomStrategy(seed=SEED),
    'Momentum': MomentumStrategy(),
}

val_results = {}
for name, agent in agents.items():
    print(f"Evaluating {name}...")
    result = evaluate_agent(agent, val_env, name, n_episodes=1, deterministic=True)
    val_results[name] = result
    
    metrics = result['metrics']
    print(f"  Sharpe: {metrics.get('sharpe_ratio', 0):.4f}")
    print(f"  Return: {metrics.get('total_return', 0):.4f}")
    print(f"  Drawdown: {metrics.get('max_drawdown', 0):.4f}")
    print()

print("✅ Validation evaluation complete")

In [ ]:
# Create comparison table
comparison_data = []
for name, result in val_results.items():
    metrics = result['metrics']
    comparison_data.append({
        'Agent': name,
        'Sharpe Ratio': metrics.get('sharpe_ratio', 0),
        'Total Return': metrics.get('total_return', 0),
        'Ann. Return': metrics.get('total_return', 0) * (252 / metrics.get('episode_length', 1)),
        'Volatility': metrics.get('volatility', 0),
        'Max Drawdown': metrics.get('max_drawdown', 0),
        'Sortino Ratio': metrics.get('sortino_ratio', 0),
        'Avg Turnover': metrics.get('mean_turnover', 0),
        'Episode Length': metrics.get('episode_length', 0),
    })

comparison_df = pd.DataFrame(comparison_data).set_index('Agent')
comparison_df = comparison_df.sort_values('Sharpe Ratio', ascending=False)

print("\n📊 VALIDATION SET PERFORMANCE COMPARISON")
print("="*100)
comparison_df.style.format({
    'Sharpe Ratio': '{:.4f}',
    'Total Return': '{:.4f}',
    'Ann. Return': '{:.4f}',
    'Volatility': '{:.4f}',
    'Max Drawdown': '{:.4f}',
    'Sortino Ratio': '{:.4f}',
    'Avg Turnover': '{:.4f}',
    'Episode Length': '{:.0f}',
})

## 5. Comparative Visualization

In [ ]:
# Plot 1: Sharpe Ratio comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Sharpe Ratio
comparison_df['Sharpe Ratio'].plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
axes[0].set_title('Sharpe Ratio Comparison', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Sharpe Ratio')
axes[0].set_xlabel('')
axes[0].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[0].grid(axis='y', alpha=0.3)
axes[0].tick_params(axis='x', rotation=45)

# Total Return
comparison_df['Total Return'].plot(kind='bar', ax=axes[1], color='green', edgecolor='black')
axes[1].set_title('Total Return Comparison', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Total Return')
axes[1].set_xlabel('')
axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[1].grid(axis='y', alpha=0.3)
axes[1].tick_params(axis='x', rotation=45)

# Max Drawdown
comparison_df['Max Drawdown'].plot(kind='bar', ax=axes[2], color='red', edgecolor='black')
axes[2].set_title('Max Drawdown Comparison', fontsize=14, fontweight='bold')
axes[2].set_ylabel('Max Drawdown')
axes[2].set_xlabel('')
axes[2].axhline(y=-0.2, color='orange', linestyle='--', alpha=0.5, label='Threshold')
axes[2].grid(axis='y', alpha=0.3)
axes[2].tick_params(axis='x', rotation=45)
axes[2].legend()

plt.suptitle('Validation Set Performance Metrics', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Plot 2: Cumulative returns
fig, ax = plt.subplots(figsize=(16, 6))

for name, result in val_results.items():
    values = result['values']
    returns = [(v / 1_000_000 - 1) * 100 for v in values]  # Convert to % return from $1M
    ax.plot(returns, label=name, linewidth=2, alpha=0.8)

ax.set_title('Cumulative Returns - Validation Set', fontsize=14, fontweight='bold')
ax.set_xlabel('Days', fontsize=12)
ax.set_ylabel('Return from Initial (%)', fontsize=12)
ax.legend(loc='best', fontsize=11)
ax.grid(alpha=0.3)
ax.axhline(y=0, color='black', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Plot 3: Risk-Return scatter
fig, ax = plt.subplots(figsize=(10, 8))

for name in comparison_df.index:
    volatility = comparison_df.loc[name, 'Volatility']
    ann_return = comparison_df.loc[name, 'Ann. Return']
    sharpe = comparison_df.loc[name, 'Sharpe Ratio']
    
    ax.scatter(volatility, ann_return, s=200, alpha=0.7, label=name)
    ax.annotate(f"{name}\n(SR={sharpe:.2f})", 
                (volatility, ann_return), 
                textcoords="offset points", 
                xytext=(0,10), 
                ha='center',
                fontsize=9)

ax.set_title('Risk-Return Profile - Validation Set', fontsize=14, fontweight='bold')
ax.set_xlabel('Annualized Volatility', fontsize=12)
ax.set_ylabel('Annualized Return', fontsize=12)
ax.grid(alpha=0.3)
ax.axhline(y=0, color='black', linestyle='--', alpha=0.3)
ax.axvline(x=0, color='black', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Portfolio Weights Analysis

Analyze how different agents allocate across assets.

In [ ]:
# Plot portfolio weights over time for each agent
n_agents = len(val_results)
fig, axes = plt.subplots(n_agents, 1, figsize=(16, 4*n_agents))

if n_agents == 1:
    axes = [axes]

for i, (name, result) in enumerate(val_results.items()):
    weights = np.array(result['weights'])
    
    # Create stacked area plot
    axes[i].stackplot(range(len(weights)), weights.T, labels=TICKERS, alpha=0.7)
    axes[i].set_title(f'{name} - Portfolio Allocation Over Time', fontsize=12, fontweight='bold')
    axes[i].set_ylabel('Weight')
    axes[i].set_ylim(0, 1)
    axes[i].legend(loc='upper left', ncol=7, fontsize=9)
    axes[i].grid(alpha=0.3)

axes[-1].set_xlabel('Days')

plt.suptitle('Portfolio Weights Evolution - Validation Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Average weights across episode
avg_weights_data = []

for name, result in val_results.items():
    weights = np.array(result['weights'])
    avg_weights = weights.mean(axis=0)
    
    for i, ticker in enumerate(TICKERS):
        avg_weights_data.append({
            'Agent': name,
            'Ticker': ticker,
            'Avg Weight': avg_weights[i]
        })

avg_weights_df = pd.DataFrame(avg_weights_data)
avg_weights_pivot = avg_weights_df.pivot(index='Agent', columns='Ticker', values='Avg Weight')

print("\n📊 AVERAGE PORTFOLIO WEIGHTS (Validation)")
print("="*80)
avg_weights_pivot.style.format('{:.4f}').background_gradient(cmap='RdYlGn', axis=1)

In [ ]:
# Visualize average weights
fig, ax = plt.subplots(figsize=(12, 6))

avg_weights_pivot.T.plot(kind='bar', ax=ax, width=0.8, edgecolor='black')
ax.set_title('Average Portfolio Weights by Agent', fontsize=14, fontweight='bold')
ax.set_xlabel('Ticker', fontsize=12)
ax.set_ylabel('Average Weight', fontsize=12)
ax.legend(title='Agent', fontsize=10)
ax.grid(axis='y', alpha=0.3)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

## 7. Test Set Evaluation

Final evaluation on held-out test set.

In [ ]:
# Evaluate on test set
print("🔍 Evaluating agents on TEST set...\n")

test_results = {}
for name, agent in agents.items():
    print(f"Evaluating {name}...")
    result = evaluate_agent(agent, test_env, name, n_episodes=1, deterministic=True)
    test_results[name] = result
    
    metrics = result['metrics']
    print(f"  Sharpe: {metrics.get('sharpe_ratio', 0):.4f}")
    print(f"  Return: {metrics.get('total_return', 0):.4f}")
    print(f"  Drawdown: {metrics.get('max_drawdown', 0):.4f}")
    print()

print("✅ Test evaluation complete")

In [ ]:
# Create test set comparison table
test_comparison_data = []
for name, result in test_results.items():
    metrics = result['metrics']
    test_comparison_data.append({
        'Agent': name,
        'Sharpe Ratio': metrics.get('sharpe_ratio', 0),
        'Total Return': metrics.get('total_return', 0),
        'Ann. Return': metrics.get('total_return', 0) * (252 / metrics.get('episode_length', 1)),
        'Volatility': metrics.get('volatility', 0),
        'Max Drawdown': metrics.get('max_drawdown', 0),
        'Sortino Ratio': metrics.get('sortino_ratio', 0),
        'Avg Turnover': metrics.get('mean_turnover', 0),
        'Episode Length': metrics.get('episode_length', 0),
    })

test_comparison_df = pd.DataFrame(test_comparison_data).set_index('Agent')
test_comparison_df = test_comparison_df.sort_values('Sharpe Ratio', ascending=False)

print("\n📊 TEST SET PERFORMANCE COMPARISON")
print("="*100)
test_comparison_df.style.format({
    'Sharpe Ratio': '{:.4f}',
    'Total Return': '{:.4f}',
    'Ann. Return': '{:.4f}',
    'Volatility': '{:.4f}',
    'Max Drawdown': '{:.4f}',
    'Sortino Ratio': '{:.4f}',
    'Avg Turnover': '{:.4f}',
    'Episode Length': '{:.0f}',
})

In [ ]:
# Plot test set cumulative returns
fig, ax = plt.subplots(figsize=(16, 6))

for name, result in test_results.items():
    values = result['values']
    returns = [(v / 1_000_000 - 1) * 100 for v in values]
    ax.plot(returns, label=name, linewidth=2, alpha=0.8)

ax.set_title('Cumulative Returns - Test Set (21 Days)', fontsize=14, fontweight='bold')
ax.set_xlabel('Days', fontsize=12)
ax.set_ylabel('Return from Initial (%)', fontsize=12)
ax.legend(loc='best', fontsize=11)
ax.grid(alpha=0.3)
ax.axhline(y=0, color='black', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## Summary & Key Findings

### 🏆 Performance Rankings (Validation Set)

Based on Sharpe Ratio:
1. **[Agent with highest Sharpe]**
2. **[Agent with second highest Sharpe]**
3. **[Agent with third highest Sharpe]**

### 📊 Key Observations

#### PPO Agent
- **Strengths:** [Based on results]
- **Weaknesses:** [Based on results]
- **Portfolio behavior:** [Allocation patterns]

#### Baseline Strategies
- **Equal Weight:** Simple, low turnover, moderate performance
- **Random:** High variance, baseline for comparison
- **Momentum:** Trend-following, higher turnover

### ⚠️ Important Notes

1. **Short Episode Length:** If agents hit max drawdown early, metrics may not be reliable
2. **Single Fold:** Results are from fold 0 only - need full walk-forward for robustness
3. **Parameter Sensitivity:** PPO performance depends on hyperparameters

### 🎯 Next Steps

1. **Increase Training:** Train PPO for 200k-500k timesteps
2. **Adjust Risk Management:** Increase max drawdown threshold for longer episodes
3. **Full Walk-Forward:** Run on all 50 folds
4. **Hyperparameter Tuning:** Optimize learning rate, gamma, entropy coefficient
5. **SAC Comparison:** Implement and compare SAC agent

### 📈 Production Readiness

**Current Status:** ⚠️ Not ready for production

**Required Improvements:**
- [ ] Longer, more stable episodes
- [ ] Validation on all 50 folds
- [ ] Risk constraints (position limits)
- [ ] Out-of-sample testing
- [ ] Regime adaptation analysis

---

**Notebook Complete** ✅